In [ ]:
import cv2 as cv
import numpy as np
from scipy.spatial.transform import Rotation
import json

rgb_json = 'intrinsics_rgb (1).json' # Input the path to the intrinsic_rgb json file
robot_file = 'calibpos.txt'

In [5]:
def quat_to_R(q):
    x,y,z,w = q
    s = np.linalg.norm([w,x,y,z])
    w,x,y,z = w/s, x/s, y/s, z/s
    R = np.array([
        [1-2*(y*y+z*z),   2*(x*y - z*w),   2*(x*z + y*w)],
        [2*(x*y + z*w),   1-2*(x*x+z*z),   2*(y*z - x*w)],
        [2*(x*z - y*w),   2*(y*z + x*w),   1-2*(x*x+y*y)]
    ])
    return R



# SHOUTOUT https://github.com/RealManRobot/hand_eye_calibration/blob/main/compute_to_hand.py
def convert(x ,y ,z, rotation_matrix, translation_vector):
    obj_camera_coordinates = np.array([x, y, z])
    T_camera_to_base_effector = np.eye(4)
    T_camera_to_base_effector[:3, :3] = rotation_matrix
    T_camera_to_base_effector[:3, 3] = translation_vector.reshape(3)

    # Compute the pose of the object against the base
    obj_camera_coordinates_homo = np.append(obj_camera_coordinates, [1])  # Convert object coordinates to homogeneous coordinates
    obj_base_effector_coordinates_homo = T_camera_to_base_effector.dot(obj_camera_coordinates_homo)
    obj_base_coordinates = obj_base_effector_coordinates_homo[:3]  

    rot_matrix_homo = T_camera_to_base_effector[:3, :3]
    quaternion = Rotation.from_matrix(rot_matrix_homo).as_quat()

    return list(obj_base_coordinates), list(quaternion)


In [ ]:
def camera2robot_calib(position_robot, quat_robot, rotation_vec_t2c, translation_t2c):
    R_g2b, t_g2b = [], []
    for pos, quat in zip(position_robot, quat_robot): #position robot list of XYZ for each position of calibration and quat is its quaternion
        R_b2g = quat_to_R(quat)                                 # (3,3)
        t_b2g = np.asarray(pos, np.float64).reshape(3,1)  #  (3,1)
        # invert to gripper->base
        R_gb = R_b2g.T
        t_gb = - R_gb @ t_b2g
        R_g2b.append(R_gb)
        t_g2b.append(t_gb)

    R_t2c = [cv.Rodrigues(np.array(rv, np.float64))[0] for rv in rotation_vec_t2c]
    t_t2c = [np.array(tv, np.float64).reshape(3,1)     for tv in translation_t2c]

    R_cam2gripper, t_cam2gripper = cv.calibrateHandEye(
        R_gripper2base=R_g2b, t_gripper2base=t_g2b,
        R_target2cam=R_t2c,   t_target2cam=t_t2c,
        method=cv.CALIB_HAND_EYE_DANIILIDIS
    )
    return R_cam2gripper, t_cam2gripper

In [ ]:
[[ 0.96225019, -0.26950574, -0.03796355],
 [ 0.25783416,  0.94733416, -0.18994613],
 [ 0.08715574,  0.17298739,  0.98106026]]

#0.05,0.02,0.10


[[ 0.96225019,  0.25783416,  0.08715574],
 [-0.26950574,  0.94733416,  0.17298739],
 [-0.03796355, -0.18994613,  0.98106026]]

#−0.06198477,−0.02277014,−0.09240893